In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from pathlib import Path
import tensorflow as tf
import torch
from torch.utils.data import Dataset, IterableDataset, DataLoader
import json
import h5py

In [ ]:
#Keys from dataset we care about
TENSOR_KEYS = [
    "global_flux_view_fluxnorm",
    #"unfolded_global_flux_view_fluxnorm",
    #"unfolded_local_flux_view_fluxnorm",
    #"local_flux_view",
    "local_flux_odd_view",
    "local_flux_even_view",
    "local_flux_view_fluxnorm",
    #"local_flux_odd_view_fluxnorm",
    #"local_flux_even_view_fluxnorm",
    "global_centr_view",
    "local_centr_view"
]
LABEL_KEYS = ["label", "TESS Disposition", "TFOPWG Disposition"]


In [ ]:
#read TFDataset file - was very complicated and used some code from ExoMiner
def tfexample_bytes_to_python(example_bytes: bytes):

    ex = tf.train.Example()
    ex.ParseFromString(example_bytes)

    out = {}
    for key, feat in ex.features.feature.items():
        kind = feat.WhichOneof("kind")
        if kind == "bytes_list":
            vals = list(feat.bytes_list.value)
            if len(vals) == 1:
                try:
                    out[key] = vals[0].decode("utf-8")
                except UnicodeDecodeError:
                    out[key] = vals[0]
            else:
                decoded = []
                for v in vals:
                    try:
                        decoded.append(v.decode("utf-8"))
                    except UnicodeDecodeError:
                        decoded.append(v)
                out[key] = decoded
        elif kind == "float_list":
            out[key] = list(feat.float_list.value)
        elif kind == "int64_list":
            out[key] = list(feat.int64_list.value)
        else:
            out[key] = None
    return out


In [ ]:
class TessShardDataset(IterableDataset):
    def __init__(self, shard_path: Path,
                 tensor_keys=None,
                 label_keys=None,
                 keep_unknowns = False):
        self.shard_path = Path(shard_path)
        self.tensor_keys = tensor_keys or []
        self.label_keys = label_keys or []
        self.keep_unknowns = keep_unknowns
        self.ds = tf.data.TFRecordDataset(str(self.shard_path))

    def __iter__(self):
        for raw in self.ds:
            ex = tfexample_bytes_to_python(raw.numpy())
            sample = {}
            # 1get tensors
            for k in self.tensor_keys:
                if k in ex and isinstance(ex[k], list) and len(ex[k]) > 0:
                    #convert to 1d torch tensor
                    sample[k] = torch.tensor(ex[k], dtype=torch.float32)
            # get first label
            label_val = None
            for lk in self.label_keys:
                if lk in ex:
                    label_val = ex[lk]
                    break
            #if we are keeping unknown lightcurves
            if(self.keep_unknowns):
              if label_val == "UNK": #is not None:
                  #label = "UNK"
                  sample["label"] = label_val
            #not keeping unknown lightcurves
            else:
              if label_val is not None and label_val != "UNK":
                  sample["label"] = label_val

            #keep uid to map back to dataset
            if "uid" in ex:
                sample["uid"] = ex["uid"]

            yield sample

    def __len__(self):

        return len(self.sample)

def collate_varlen(batch):
    out = {}
    for sample in batch:
        for k, v in sample.items():
            out.setdefault(k, []).append(v)
    return out


In [ ]:
def build_h5_from_shards(
    shard_paths,
    h5_path,
    tensor_keys,
    label_keys,
    keep_unknowns=False,
):

    shard_paths = [Path(p) for p in shard_paths]
    #split labels into binary labels
    NP_labels = ['NTP', 'EB', 'FP', 'NEB']
    P_labels = ["CP", "KP"]

    #mapping
    label_to_idx = {"NP": 0, "P": 1, "UNK": 2}
    label_counts = {"NP": 0, "P": 0, "UNK": 0}
    next_label_id = 0
    n_examples = 0

    with h5py.File(h5_path, "w") as f:
        f.attrs["tensor_keys"] = json.dumps(tensor_keys)
        f.attrs["label_keys"] = json.dumps(label_keys)

        dsets = {}
        dset_label = None
        dset_uid = None

        for shard_path in shard_paths:
            print(f"Reading shard: {shard_path}")
            ds = TessShardDataset(
                shard_path,
                tensor_keys=tensor_keys,
                label_keys=label_keys,
                keep_unknowns=keep_unknowns,
            )

            for sample in ds:
                if "label" not in sample:
                    continue

                label_str = sample["label"]
                #if(label_str == "UNK"):
                #    continue

                if(label_str in NP_labels):
                    label_counts["NP"] += 1
                    label_id = label_to_idx["NP"]
                elif(label_str in P_labels):
                    label_counts["P"] += 1
                    label_id = label_to_idx["P"]
                elif(label_str == "UNK"):
                    label_counts["UNK"] += 1
                    label_id = label_to_idx["UNK"]

                """
                if label_str not in label_to_idx:
                    label_to_idx[label_str] = next_label_id
                    next_label_id += 1
                label_id = label_to_idx[label_str]

                """
                # Convert tensors to numpy
                np_tensors = {}
                for k in tensor_keys:
                    if k in sample:
                        arr = sample[k].detach().cpu().numpy()
                        #1D
                        np_tensors[k] = arr

                if not dsets:
                    for k, arr in np_tensors.items():
                        L = arr.shape[0]
                        dsets[k] = f.create_dataset(
                            k,
                            shape=(0, L),
                            maxshape=(None, L),
                            dtype="float32",
                            chunks=True,
                        )
                    dset_label = f.create_dataset(
                        "label",
                        shape=(0,),
                        maxshape=(None,),
                        dtype="int64",
                        chunks=True,
                    )
                    if "uid" in sample:
                        dset_uid = f.create_dataset(
                            "uid",
                            shape=(0,),
                            maxshape=(None,),
                            dtype=h5py.string_dtype(encoding="utf-8"),
                            chunks=True,
                        )

                old_n = dset_label.shape[0]
                new_n = old_n + 1

                # Resize datasets
                for k, d in dsets.items():
                    d.resize((new_n, d.shape[1]))
                dset_label.resize((new_n,))
                if dset_uid is not None:
                    dset_uid.resize((new_n,))

                # Write data
                for k, arr in np_tensors.items():
                    dsets[k][old_n] = arr
                dset_label[old_n] = label_id
                if dset_uid is not None:
                    dset_uid[old_n] = str(sample.get("uid", ""))

                n_examples += 1

        f.attrs["label_to_idx"] = json.dumps(label_to_idx)

    print(f"Saved {n_examples} examples to {h5_path}")
    print("label_to_idx mapping:", label_to_idx)
    print("label_counts:", label_counts)

In [ ]:
SHARD_DIR = Path("drive/MyDrive/Exoplanet Detection/shards")
shard_paths = sorted(SHARD_DIR.glob("shard-*"))

#TENSOR_KEYS = [
#    "global_flux_view_fluxnorm",
#    "local_flux_view_fluxnorm",
#]
LABEL_KEYS = ["label"]
#build combined dataset of shards - use keep_unknowns = True to get SSL set
build_h5_from_shards(
    shard_paths=shard_paths,
    h5_path="drive/MyDrive/Exoplanet Detection/tess_lightcurves_not_unk.h5",
    tensor_keys=TENSOR_KEYS,
    label_keys=LABEL_KEYS,
    keep_unknowns=False,
)

In [ ]:
SHARD_DIR = Path("drive/MyDrive/Exoplanet Detection/shards")
shard_paths = sorted(SHARD_DIR.glob("shard-*"))

#TENSOR_KEYS = [
#    "global_flux_view_fluxnorm",
#    "local_flux_view_fluxnorm",
#]
LABEL_KEYS = ["label"]
#build combined dataset of shards - use keep_unknowns = True to get SSL set
build_h5_from_shards(
    shard_paths=shard_paths,
    h5_path="drive/MyDrive/Exoplanet Detection/tess_lightcurves_unk.h5",
    tensor_keys=TENSOR_KEYS,
    label_keys=LABEL_KEYS,
    keep_unknowns=True,
)

In [ ]:
import h5py
import torch
from torch.utils.data import Dataset

class TessH5Dataset(Dataset):
    def __init__(self, h5_path, use_local=True, use_global=True, transform=None):
        self.h5_path = h5_path
        self._f = h5py.File(h5_path, "r")
        self.use_global = use_global
        self.use_local = use_local
        self.transform = transform

        # Required
        self.labels = self._f["label"]

        # Optional inputs
        self.global_flux = self._f["global_flux_view_fluxnorm"] if use_global else None
        self.global_centr = self._f["global_centr_view"] if use_global else None
        self.local_centr = self._f["local_centr_view"] if use_global else None
        self.local_flux = self._f["local_flux_view_fluxnorm"] if use_local else None

    def __len__(self):
        return self.labels.shape[0]

    def __getitem__(self, idx):
        x = {}
        if self.global_flux is not None:
            # For 1D CNN: shape (C=1, L)
            x["global"] = torch.from_numpy(self.global_flux[idx]).unsqueeze(0)
        if self.local_flux is not None:
            x["local"] = torch.from_numpy(self.local_flux[idx]).unsqueeze(0)

        y = torch.tensor(self.labels[idx], dtype=torch.long)

        if self.transform is not None:
            x = self.transform(x)

        return x, y

    def close(self):
        self._f.close()


In [ ]:
import h5py
import json
import numpy as np
import matplotlib.pyplot as plt

H5_PATH = "drive/MyDrive/Exoplanet Detection/tess_lightcurves_unk.h5"
IDX = 0

with h5py.File(H5_PATH, "r") as f:

    print("Datasets;")
    for name in f.keys():
        print(f"  {name}: shape={f[name].shape}, dtype={f[name].dtype}")


    if "label_to_idx" in f.attrs:
        label_to_idx = json.loads(f.attrs["label_to_idx"])
        idx_to_label = {v: k for k, v in label_to_idx.items()}
        print("label_to_idx:", label_to_idx)
    else:
        label_to_idx = {}
        idx_to_label = {}

    #load flux views
    lc_global = f["global_flux_view_fluxnorm"][IDX]
    print("Global LC shape:", lc_global.shape)

    lc_local = None
    if "local_flux_view_fluxnorm" in f:
        lc_local = f["local_flux_view_fluxnorm"][IDX]
        print("Local LC shape:", lc_local.shape)

    #load centroids
    centr_global = None
    if "global_centr_view" in f:
        centr_global = f["global_centr_view"][IDX]
        print("Global centroid shape:", centr_global.shape)

    centr_local = None
    if "local_centr_view" in f:
        centr_local = f["local_centr_view"][IDX]
        print("Local centroid shape:", centr_local.shape)


    #get label
    if "label" in f:
        y = f["label"][IDX]
        if idx_to_label:
            print(f"Example {IDX} label:", y,
                  f"({idx_to_label.get(int(y),'unknown')})")
        else:
            print(f"Example {IDX} label:", y)

#PLOTTING

#global flux plot
plt.figure(figsize=(10, 4))
plt.plot(np.arange(len(lc_global)), lc_global)
plt.xlabel("Time index")
plt.ylabel("Normalized flux (global)")
plt.title(f"Global flux light curve (example {IDX})")
plt.tight_layout()
plt.show()

#local flux plot
if lc_local is not None:
    plt.figure(figsize=(10, 4))
    plt.plot(np.arange(len(lc_local)), lc_local)
    plt.xlabel("Time index")
    plt.ylabel("Normalized flux (local)")
    plt.title(f"Local flux light curve (example {IDX})")
    plt.tight_layout()
    plt.show()

#global centroid plot
if centr_global is not None:
    plt.figure(figsize=(10, 4))
    plt.plot(np.arange(len(centr_global)), centr_global)
    plt.xlabel("Time index")
    plt.ylabel("Centroid (global)")
    plt.title(f"Global centroid time series (example {IDX})")
    plt.tight_layout()
    plt.show()

#local centroid plot
if centr_local is not None:
    plt.figure(figsize=(10, 4))
    plt.plot(np.arange(len(centr_local)), centr_local)
    plt.xlabel("Time index")
    plt.ylabel("Centroid (local)")
    plt.title(f"Local centroid time series (example {IDX})")
    plt.tight_layout()
    plt.show()
